"""
Improved RNA 3D Structure Prediction - Kaggle Offline Optimized
================================================================
⚠️ INTERNET DISABLED - Uses only standard library and pre-installed packages
"""

# ============================================================================
# IMPORTS - Only standard Kaggle packages (no external downloads)
# ============================================================================

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.optimize import minimize
import random
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("="*70)
print("RNA 3D STRUCTURE PREDICTION - KAGGLE OFFLINE MODE")
print("="*70)
print("Internet: DISABLED ✓")
print("Using: NumPy, SciPy, Pandas (pre-installed)")
print("="*70)

# ============================================================================
# CONFIGURATION
# ============================================================================

In [ ]:
# RNA geometry constants (in Angstroms)
PHOSPHATE_DISTANCE = 5.9
BASE_PAIR_DISTANCE = 10.5
STACK_DISTANCE = 3.4
BACKBONE_RISE = 2.8

# Base pairing rules
WATSON_CRICK = {'A': 'U', 'U': 'A', 'G': 'C', 'C': 'G'}
WOBBLE_PAIRS = {('G', 'U'), ('U', 'G')}

# ============================================================================
# DATA LOADING
# ============================================================================

In [ ]:
print("\n📂 Loading competition data...")
try:
    # These paths are standard for Kaggle competitions
    DATA_PATH = '/kaggle/input/stanford-rna-3d-folding/'
    
    test_seqs = pd.read_csv(DATA_PATH + 'test_sequences.csv')
    train_seqs = pd.read_csv(DATA_PATH + 'train_sequences.csv')
    validation_seqs = pd.read_csv(DATA_PATH + 'validation_sequences.csv')
    train_labels = pd.read_csv(DATA_PATH + 'train_labels.csv')
    validation_labels = pd.read_csv(DATA_PATH + 'validation_labels.csv')
    sample_submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')
    
    print(f"✅ Train sequences: {len(train_seqs)}")
    print(f"✅ Validation sequences: {len(validation_seqs)}")
    print(f"✅ Test sequences: {len(test_seqs)}")
    
    # Use validation for development, test for final submission
    TARGET_SEQS = validation_seqs  # Change to test_seqs for submission
    TARGET_LABELS = validation_labels  # For scoring during development
    
except FileNotFoundError as e:
    print(f"⚠️ Competition data not found: {e}")
    print("⚠️ Creating minimal test data...")
    TARGET_SEQS = pd.DataFrame({
        'target_id': ['TEST1', 'TEST2'],
        'sequence': ['GGGGCCCCAAAA', 'AUCGAUCG']
    })
    train_seqs = pd.DataFrame({'target_id': [], 'sequence': []})
    train_labels = pd.DataFrame()
    TARGET_LABELS = None

# ============================================================================
# SECONDARY STRUCTURE PREDICTION (Nussinov Algorithm)
# ============================================================================

In [ ]:
def nussinov_fold(sequence, min_loop_size=3):
    """
    Nussinov dynamic programming algorithm for RNA secondary structure.
    Returns list of base pairs (i, j) where i < j.
    Time complexity: O(n^3), Space: O(n^2)
    """
    n = len(sequence)
    if n == 0:
        return []
    
    # DP table and traceback
    dp = np.zeros((n, n), dtype=int)
    traceback = {}
    
    def can_pair(i, j):
        if j - i <= min_loop_size:
            return False
        pair = (sequence[i], sequence[j])
        return (sequence[i] in WATSON_CRICK and 
                WATSON_CRICK[sequence[i]] == sequence[j]) or pair in WOBBLE_PAIRS
    
    # Fill DP table diagonally
    for length in range(min_loop_size + 1, n):
        for i in range(n - length):
            j = i + length
            
            # Option 1: j is unpaired
            dp[i][j] = dp[i][j-1]
            traceback[(i, j)] = ('unpaired', j)
            
            # Option 2: (i,j) forms a base pair
            if can_pair(i, j):
                score = (dp[i+1][j-1] if i+1 <= j-1 else 0) + 1
                if score > dp[i][j]:
                    dp[i][j] = score
                    traceback[(i, j)] = ('pair', i, j)
            
            # Option 3: bifurcation at some k
            for k in range(i + 1, j):
                score = dp[i][k] + dp[k+1][j]
                if score > dp[i][j]:
                    dp[i][j] = score
                    traceback[(i, j)] = ('bifurc', k)
    
    # Traceback to extract base pairs
    def trace(i, j, pairs):
        if i >= j or (i, j) not in traceback:
            return
        
        action = traceback[(i, j)]
        if action[0] == 'unpaired':
            trace(i, j-1, pairs)
        elif action[0] == 'pair':
            pairs.append((i, j))
            if i+1 <= j-1:
                trace(i+1, j-1, pairs)
        elif action[0] == 'bifurc':
            k = action[1]
            trace(i, k, pairs)
            trace(k+1, j, pairs)
    
    pairs = []
    trace(0, n-1, pairs)
    return sorted(pairs)

def find_stems(base_pairs):
    """Group consecutive base pairs into stem-loop structures."""
    if not base_pairs:
        return []
    
    stems = []
    current_stem = [base_pairs[0]]
    
    for i in range(1, len(base_pairs)):
        prev_i, prev_j = base_pairs[i-1]
        curr_i, curr_j = base_pairs[i]
        
        # Consecutive pairs form a stem
        if curr_i == prev_i + 1 and curr_j == prev_j - 1:
            current_stem.append(base_pairs[i])
        else:
            stems.append(current_stem)
            current_stem = [base_pairs[i]]
    
    stems.append(current_stem)
    return stems

# ============================================================================
# TEMPLATE-BASED MODELING (Sequence Similarity)
# ============================================================================

In [ ]:
def sequence_similarity(seq1, seq2):
    """Calculate pairwise sequence identity."""
    if len(seq1) == 0 or len(seq2) == 0:
        return 0.0
    min_len = min(len(seq1), len(seq2))
    matches = sum(1 for a, b in zip(seq1[:min_len], seq2[:min_len]) if a == b)
    return matches / min_len

def find_best_template(target_seq, train_seqs_df, train_labels_df, min_identity=0.35):
    """Find most similar sequence in training set."""
    if len(train_seqs_df) == 0:
        return None, 0.0
    
    best_match = None
    best_score = 0.0
    
    for _, row in train_seqs_df.iterrows():
        sim = sequence_similarity(target_seq, row['sequence'])
        
        if sim > best_score and sim >= min_identity:
            best_score = sim
            best_match = row['target_id']
    
    return best_match, best_score

def extract_template_coords(template_id, train_labels_df, model_num=1):
    """Extract 3D coordinates from training labels."""
    if len(train_labels_df) == 0:
        return []
    
    template_data = train_labels_df[
        train_labels_df['ID'].str.startswith(template_id)
    ].sort_values('resid')
    
    coords = []
    for _, row in template_data.iterrows():
        x = row[f'x_{model_num}']
        y = row[f'y_{model_num}']
        z = row[f'z_{model_num}']
        
        # Check for missing coordinates (-1e18 sentinel)
        if x < -1e17:
            coords.append(None)
        else:
            coords.append(np.array([x, y, z], dtype=np.float64))
    
    return coords

# ============================================================================
# 3D STRUCTURE BUILDER
# ============================================================================

In [ ]:
class RNAStructureBuilder:
    """Build RNA 3D structure using physics-based rules."""
    
    def __init__(self, sequence, secondary_structure=None, seed=None):
        self.sequence = sequence
        self.n = len(sequence)
        self.ss = secondary_structure or []
        self.coords = np.zeros((self.n, 3), dtype=np.float64)
        
        if seed is not None:
            np.random.seed(seed)
            random.seed(seed)
    
    def build_helix(self, start_i, end_i, start_j, end_j, origin, direction):
        """Build A-form RNA double helix."""
        helix_len = min(end_i - start_i, end_j - start_j) + 1
        
        if helix_len <= 0:
            return
        
        # A-form RNA geometry
        rise_per_bp = 2.81  # Angstroms
        rotation_per_bp = 32.7 * np.pi / 180  # radians
        
        # Create perpendicular vector
        perp = np.array([-direction[1], direction[0], 0])
        if np.linalg.norm(perp) < 0.1:
            perp = np.array([0, -direction[2], direction[1]])
        perp = perp / (np.linalg.norm(perp) + 1e-10)
        
        for k in range(helix_len):
            idx1 = start_i + k
            idx2 = end_j - k
            
            if idx1 >= self.n or idx2 < 0 or idx2 >= self.n:
                break
            
            angle = k * rotation_per_bp
            rot_matrix = R.from_rotvec(angle * direction).as_matrix()
            
            # Position both strands
            offset1 = rot_matrix @ (BASE_PAIR_DISTANCE / 2 * perp)
            offset2 = rot_matrix @ (-BASE_PAIR_DISTANCE / 2 * perp)
            
            self.coords[idx1] = origin + k * rise_per_bp * direction + offset1
            self.coords[idx2] = origin + k * rise_per_bp * direction + offset2
    
    def build_loop(self, start, end, anchor_start, anchor_end):
        """Build single-stranded loop region."""
        if start >= self.n or end < 0 or start > end:
            return
        
        end = min(end, self.n - 1)
        loop_len = end - start + 1
        
        if loop_len <= 0:
            return
        
        # Smooth interpolation with curvature
        mid_point = (anchor_start + anchor_end) / 2
        mid_offset = np.random.randn(3) * 3.0
        control_point = mid_point + mid_offset
        
        for k in range(loop_len):
            idx = start + k
            if idx >= self.n:
                break
            
            t = k / max(1, loop_len - 1)
            # Quadratic Bezier curve
            self.coords[idx] = ((1-t)**2 * anchor_start + 
                               2*(1-t)*t * control_point + 
                               t**2 * anchor_end)
    
    def build_secondary_structure_guided(self):
        """Build structure guided by secondary structure prediction."""
        if self.n == 0:
            return self.coords
        
        stems = find_stems(self.ss)
        
        if not stems:
            return self.build_extended()
        
        # Initialize at origin
        origin = np.array([0.0, 0.0, 0.0])
        direction = np.array([0.0, 0.0, 1.0])
        
        # Build each stem
        for stem_idx, stem in enumerate(stems):
            start_i, start_j = stem[0]
            end_i, end_j = stem[-1]
            
            if stem_idx > 0:
                # Update position from previous stem
                prev_stem = stems[stem_idx - 1]
                prev_end_i = prev_stem[-1][0]
                
                if prev_end_i < self.n:
                    origin = self.coords[prev_end_i].copy()
                
                # Change direction for new stem
                angle = np.random.uniform(0.4, 0.8)
                axis = np.random.randn(3)
                axis = axis / (np.linalg.norm(axis) + 1e-10)
                rot = R.from_rotvec(angle * axis)
                direction = rot.apply(direction)
            
            self.build_helix(start_i, end_i, start_j, end_j, origin, direction)
            
            if end_i < self.n:
                origin = self.coords[end_i].copy()
        
        # Fill unpaired regions (loops)
        paired = set()
        for i, j in self.ss:
            paired.add(i)
            paired.add(j)
        
        i = 0
        while i < self.n:
            if i not in paired:
                j = i
                while j < self.n and j not in paired:
                    j += 1
                
                anchor_start = self.coords[i-1] if i > 0 else np.array([0., 0., 0.])
                anchor_end = (self.coords[j] if j < self.n else 
                             (self.coords[i-1] + np.array([5., 0., 0.]) if i > 0 
                              else np.array([5., 0., 0.])))
                
                self.build_loop(i, min(j-1, self.n-1), anchor_start, anchor_end)
                i = j
            else:
                i += 1
        
        return self.coords
    
    def build_extended(self):
        """Build extended random coil structure."""
        if self.n == 0:
            return self.coords
        
        direction = np.array([0.0, 0.0, 1.0])
        
        for i in range(self.n):
            if i < 3:
                # Initial helix-like start
                angle = i * 0.6
                self.coords[i] = [10.0 * np.cos(angle), 
                                 10.0 * np.sin(angle), 
                                 i * 2.5]
            else:
                # Random walk with directional persistence
                if random.random() < 0.25:
                    angle = random.uniform(0.3, 0.6)
                    axis = np.random.randn(3)
                    axis = axis / (np.linalg.norm(axis) + 1e-10)
                    rot = R.from_rotvec(angle * axis)
                    direction = rot.apply(direction)
                
                # Small random perturbations
                direction += np.random.randn(3) * 0.12
                direction = direction / (np.linalg.norm(direction) + 1e-10)
                
                # Step forward
                step = random.uniform(3.5, 4.5)
                self.coords[i] = self.coords[i-1] + step * direction
        
        return self.coords
    
    def build_from_template(self, template_coords):
        """Build structure from template with gap filling."""
        # Copy valid template coordinates
        for i in range(min(self.n, len(template_coords))):
            if template_coords[i] is not None:
                self.coords[i] = template_coords[i].copy()
        
        # Fill gaps via linear interpolation
        for i in range(self.n):
            if i >= len(template_coords) or template_coords[i] is None:
                prev_valid = next_valid = None
                
                for j in range(i-1, -1, -1):
                    if j < len(template_coords) and template_coords[j] is not None:
                        prev_valid = j
                        break
                
                for j in range(i+1, min(self.n, len(template_coords))):
                    if template_coords[j] is not None:
                        next_valid = j
                        break
                
                if prev_valid is not None and next_valid is not None:
                    t = (i - prev_valid) / (next_valid - prev_valid)
                    self.coords[i] = ((1-t) * template_coords[prev_valid] + 
                                     t * template_coords[next_valid])
                elif prev_valid is not None:
                    self.coords[i] = (template_coords[prev_valid] + 
                                     np.random.randn(3) * 4.0)
                else:
                    self.coords[i] = np.random.randn(3) * 10.0
        
        return self.coords

# ============================================================================
# ENSEMBLE GENERATION
# ============================================================================

In [ ]:
def generate_ensemble(sequence, target_id=None, n_models=5):
    """Generate diverse ensemble of 5 structure predictions."""
    structures = []
    
    # Model 1-2: Secondary structure guided (different seeds)
    ss = nussinov_fold(sequence)
    for i in range(2):
        builder = RNAStructureBuilder(sequence, ss, seed=SEED+i*100)
        coords = builder.build_secondary_structure_guided()
        structures.append(coords)
    
    # Model 3: Template-based (if available)
    if target_id and len(train_seqs) > 0:
        template_id, similarity = find_best_template(
            sequence, train_seqs, train_labels, min_identity=0.35
        )
        
        if template_id and similarity > 0.35:
            template_coords = extract_template_coords(template_id, train_labels, 1)
            if template_coords:
                builder = RNAStructureBuilder(sequence, seed=SEED+200)
                coords = builder.build_from_template(template_coords)
                structures.append(coords)
            else:
                builder = RNAStructureBuilder(sequence, seed=SEED+200)
                coords = builder.build_extended()
                structures.append(coords)
        else:
            builder = RNAStructureBuilder(sequence, seed=SEED+200)
            coords = builder.build_extended()
            structures.append(coords)
    else:
        builder = RNAStructureBuilder(sequence, seed=SEED+200)
        coords = builder.build_extended()
        structures.append(coords)
    
    # Model 4-5: Extended conformations
    for i in range(2):
        builder = RNAStructureBuilder(sequence, seed=SEED+300+i*100)
        coords = builder.build_extended()
        structures.append(coords)
    
    return structures[:n_models]

# ============================================================================
# SUBMISSION BUILDER
# ============================================================================

In [ ]:
def build_submission_dataframe(sequences_df, predictions_dict):
    """Convert predictions to submission format."""
    rows = []
    
    for _, row in sequences_df.iterrows():
        target_id = row['target_id']
        sequence = row['sequence']
        
        if target_id not in predictions_dict:
            continue
        
        structures = predictions_dict[target_id]
        
        for j in range(len(sequence)):
            pred_row = {
                'ID': f"{target_id}_{j+1}",
                'resname': sequence[j],
                'resid': j + 1
            }
            
            # Add coordinates from all 5 models
            for i in range(5):
                model_idx = min(i, len(structures) - 1)
                pred_row[f'x_{i+1}'] = structures[model_idx][j][0]
                pred_row[f'y_{i+1}'] = structures[model_idx][j][1]
                pred_row[f'z_{i+1}'] = structures[model_idx][j][2]
            
            rows.append(pred_row)
    
    return pd.DataFrame(rows)

# ============================================================================
# MAIN PREDICTION LOOP
# ============================================================================

In [ ]:
print("\n🔬 Generating structure predictions...")
print(f"Processing {len(TARGET_SEQS)} sequences...")

predictions = {}

for idx, row in TARGET_SEQS.iterrows():
    target_id = row['target_id']
    sequence = row['sequence']
    
    # Progress indicator
    if (idx + 1) % 5 == 0 or idx < 3 or idx == len(TARGET_SEQS) - 1:
        print(f"  [{idx+1}/{len(TARGET_SEQS)}] {target_id} ({len(sequence)} nt)")
    
    try:
        structures = generate_ensemble(sequence, target_id, n_models=5)
        predictions[target_id] = structures
    except Exception as e:
        print(f"  ⚠️ Error on {target_id}: {e}")
        # Fallback: random structure
        fallback = np.random.randn(len(sequence), 3) * 10.0
        predictions[target_id] = [fallback] * 5

# ============================================================================
# CREATE SUBMISSION FILE
# ============================================================================

In [ ]:
print("\n📝 Building submission file...")
submission_df = build_submission_dataframe(TARGET_SEQS, predictions)

# Ensure correct column order
columns = ['ID', 'resname', 'resid']
for i in range(1, 6):
    columns.extend([f'x_{i}', f'y_{i}', f'z_{i}'])

submission_df = submission_df[columns]
submission_df.to_csv('submission.csv', index=False)

print(f"✅ Submission created: {len(submission_df)} rows")
print(f"✅ Sequences processed: {len(predictions)}")
print("\n📊 Sample output:")
print(submission_df.head(10))

# ============================================================================
# EVALUATION (if validation labels available)
# ============================================================================

In [ ]:
if TARGET_LABELS is not None and len(TARGET_LABELS) > 0:
    print("\n" + "="*70)
    print("EVALUATION")
    print("="*70)
    
    try:
        import sys
        sys.path.append('/kaggle/usr/lib/ribonanza-tm-score')
        from metric import score
        
        submission_df['target_id'] = submission_df['ID'].str.split('_').str[0]
        tm_score = score(TARGET_LABELS, submission_df.copy(), row_id_column_name='ID')
        
        print(f"\n🎯 TM-score: {tm_score:.4f}")
        
        if tm_score > 0.20:
            print("   ✅ Good - Above baseline!")
        elif tm_score > 0.15:
            print("   ⚠️ Fair - Near baseline")
        else:
            print("   ⚠️ Low - Check predictions")
            
    except Exception as e:
        print(f"⚠️ Could not evaluate: {e}")

print("\n" + "="*70)
print("✅ COMPLETE - submission.csv ready for upload")
print("="*70)